In [1]:
import sys; sys.path.append("../src")
import torch
from data_prep import prepare_data
from models import IDSNet
from attacks import pgd

d = prepare_data("../data/KDDTrain.txt", "../data/KDDTest.txt")
X_test = torch.tensor(d["X_test"]); y_test = torch.tensor(d["y_test"], dtype=torch.long)

CONTROLLABLE = ["duration", "src_bytes", "count", "srv_count", "dst_host_count", "dst_host_srv_count"]
mask = torch.tensor([1.0 if n in CONTROLLABLE else 0.0 for n in d["feature_names"]])

def load(path):
    m = IDSNet(X_test.shape[1]); m.load_state_dict(torch.load(path)); m.eval(); return m

models = {"baseline": load("../saved_models/nn_baseline.pt"),
          "hardened": load("../saved_models/nn_hardened.pt")}

In [2]:
print(f"{'model':>9} | {'test acc':>8} | {'attack recall':>13}")
for n, m in models.items():
    with torch.no_grad():
        p = m(X_test).argmax(dim=1)
    print(f"{n:>9} | {(p == y_test).float().mean().item():>8.3f} | {(p[y_test == 1] == 1).float().mean().item():>13.3f}")

    model | test acc | attack recall
 baseline |    0.780 |         0.667
 hardened |    0.779 |         0.643


In [3]:
def evasion(model, eps, mask=None):
    with torch.no_grad():
        caught = model(X_test).argmax(dim=1) == 1
    m = (y_test == 1) & caught
    Xa = pgd(model, X_test[m], y_test[m], eps, mask=mask)
    with torch.no_grad():
        return (model(Xa).argmax(dim=1) == 0).float().mean().item() * 100

print(f"{'eps':>5} | {'base PGD':>9} | {'hard PGD':>9} | {'base realistic':>14} | {'hard realistic':>14}")
for eps in [0.05, 0.1, 0.2, 0.5, 1.0]:
    print(f"{eps:>5.2f} | {evasion(models['baseline'], eps):>8.1f}% | {evasion(models['hardened'], eps):>8.1f}% | "
          f"{evasion(models['baseline'], eps, mask):>13.1f}% | {evasion(models['hardened'], eps, mask):>13.1f}%")

  eps |  base PGD |  hard PGD | base realistic | hard realistic
 0.05 |     40.2% |      7.2% |           1.4% |           1.2%
 0.10 |     98.7% |     13.2% |           3.0% |           1.8%
 0.20 |     99.9% |     22.9% |           7.4% |           3.3%
 0.50 |    100.0% |     98.0% |          17.8% |           8.3%
 1.00 |    100.0% |    100.0% |          26.8% |          16.5%
